# 🦺 PPE Compliance Detection with YOLOv8
**Dataset:** [Simuletic CCTV-PPE Compliance Dataset](https://github.com/Simuletic/cctv-ppe-compliance-detection-dataset)  
**Model:** YOLOv8 (Ultralytics)  
**Task:** Detect PPE violations (missing/wrong helmet, vest, gloves)

---
### 9 Detection Classes
| ID | Class | Violation? |
|----|-------|------------|
| 0 | person | — |
| 1 | hardhat | ✅ Compliant |
| 2 | hardhat_wrong | ⚠️ Violation |
| 3 | no_hardhat | ❌ Violation |
| 4 | vest | ✅ Compliant |
| 5 | vest_wrong | ⚠️ Violation |
| 6 | no_vest | ❌ Violation |
| 7 | gloves | ✅ Compliant |
| 8 | no_gloves | ❌ Violation |

## 0 · Environment Setup

In [ ]:
# Check GPU availability
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('PyTorch version:', torch.__version__)

In [ ]:
# Install dependencies (run once)
# If running on Colab / Kaggle, uncomment:
# !pip install ultralytics opencv-python-headless matplotlib seaborn -q

# Local install:
# !pip install -r ../requirements.txt

## 1 · Clone Dataset & Prepare Splits

In [ ]:
import os, sys
# Navigate to project root so relative paths work
os.chdir('..')          # from notebooks/ → project root
sys.path.insert(0, '.')
print('Working dir:', os.getcwd())

In [ ]:
# Clone dataset into dataset/raw/
REPO = 'https://github.com/Simuletic/cctv-ppe-compliance-detection-dataset.git'
!git clone {REPO} dataset/raw/

In [ ]:
# Prepare train/val/test splits
%run scripts/prepare_dataset.py --val_ratio 0.15 --test_ratio 0.05

In [ ]:
# Quick sanity check
from pathlib import Path
for split in ('train', 'val', 'test'):
    imgs = list(Path(f'dataset/images/{split}').glob('*.*'))
    lbls = list(Path(f'dataset/labels/{split}').glob('*.txt'))
    print(f'{split:5s}: {len(imgs):5d} images | {len(lbls):5d} labels')

## 2 · Explore the Dataset

In [ ]:
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path

CLASS_NAMES = ['person','hardhat','hardhat_wrong','no_hardhat',
               'vest','vest_wrong','no_vest','gloves','no_gloves']
COLORS = plt.cm.tab10.colors

def show_sample(split='train', n=4):
    imgs = list(Path(f'dataset/images/{split}').glob('*.*'))
    sample = random.sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(1, len(sample), figsize=(5*len(sample), 5))
    if len(sample) == 1: axes = [axes]
    for ax, img_path in zip(axes, sample):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = Path(f'dataset/labels/{split}') / (img_path.stem + '.txt')
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                cls, cx, cy, bw, bh = map(float, line.split())
                cls = int(cls)
                x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                color = tuple(int(c*255) for c in COLORS[cls % 10])
                cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                cv2.putText(img, CLASS_NAMES[cls], (x1,y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')
    plt.tight_layout(); plt.show()

show_sample('train', n=4)

In [ ]:
# Class distribution in training set
import collections, matplotlib.pyplot as plt
from pathlib import Path

counter = collections.Counter()
for lbl in Path('dataset/labels/train').glob('*.txt'):
    for line in lbl.read_text().splitlines():
        if line.strip():
            cls = int(line.split()[0])
            counter[CLASS_NAMES[cls]] += 1

plt.figure(figsize=(10, 4))
bars = plt.bar(counter.keys(), counter.values(),
               color=['#2ecc71' if 'no_' not in k and 'wrong' not in k else '#e74c3c'
                      for k in counter.keys()])
plt.title('Class Distribution – Training Set')
plt.xlabel('Class'); plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
for bar, v in zip(bars, counter.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 3 · Train YOLOv8

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# ── Configuration ──────────────────────────────────────────
MODEL_VARIANT = 'yolov8n'   # n / s / m / l / x
EPOCHS        = 50
BATCH_SIZE    = 16          # reduce to 8 if OOM
IMG_SIZE      = 640
DEVICE        = 0           # GPU index; use 'cpu' if no GPU
DATA_YAML     = Path('dataset/ppe_dataset.yaml').resolve()
# ───────────────────────────────────────────────────────────

model = YOLO(f'{MODEL_VARIANT}.pt')   # downloads pretrained COCO weights

results = model.train(
    data         = str(DATA_YAML),
    epochs       = EPOCHS,
    batch        = BATCH_SIZE,
    imgsz        = IMG_SIZE,
    device       = DEVICE,
    project      = 'runs/train',
    name         = f'ppe_{MODEL_VARIANT}',
    exist_ok     = True,
    patience     = 20,
    optimizer    = 'AdamW',
    lr0          = 1e-3,
    warmup_epochs= 3,
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.1,
    plots        = True,
    save_period  = 10,
    val          = True,
)

print('Best weights:', f'runs/train/ppe_{MODEL_VARIANT}/weights/best.pt')

## 4 · Training Curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

log_csv = Path(f'runs/train/ppe_{MODEL_VARIANT}/results.csv')
df = pd.read_csv(log_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plots = [
    ('train/box_loss', 'val/box_loss', 'Box Loss'),
    ('train/cls_loss', 'val/cls_loss', 'Class Loss'),
    ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss'),
    ('metrics/precision(B)', 'metrics/recall(B)', 'Precision / Recall'),
    ('metrics/mAP50(B)',  None, 'mAP@50'),
    ('metrics/mAP50-95(B)', None, 'mAP@50-95'),
]
for ax, (col1, col2, title) in zip(axes.flat, plots):
    if col1 in df: ax.plot(df[col1], label=col1.split('/')[-1])
    if col2 and col2 in df: ax.plot(df[col2], label=col2.split('/')[-1])
    ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
plt.suptitle(f'Training Curves – ppe_{MODEL_VARIANT}', fontsize=14)
plt.tight_layout(); plt.show()

## 5 · Evaluate on Test Set

In [ ]:
best_weights = f'runs/train/ppe_{MODEL_VARIANT}/weights/best.pt'
eval_model   = YOLO(best_weights)

metrics = eval_model.val(
    data     = str(DATA_YAML),
    split    = 'test',
    imgsz    = IMG_SIZE,
    device   = DEVICE,
    plots    = True,
    save_json= True,
)

print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall   : {metrics.box.mr:.4f}')

## 6 · Inference & Violation Summary

In [ ]:
import cv2, matplotlib.pyplot as plt
from pathlib import Path
from utils.visualize import draw_boxes, VIOLATION_IDS, CLASS_NAMES

VIOLATION_CLASSES = {2:'hardhat_wrong', 3:'no_hardhat',
                     5:'vest_wrong', 6:'no_vest', 8:'no_gloves'}

test_imgs = list(Path('dataset/images/test').glob('*.*'))[:6]
results   = eval_model.predict(test_imgs, conf=0.4, iou=0.45,
                                imgsz=IMG_SIZE, device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, r in zip(axes.flat, results):
    img    = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    boxes  = r.boxes.xyxy.cpu().numpy()
    cls_ids= r.boxes.cls.int().cpu().tolist()
    confs  = r.boxes.conf.cpu().tolist()
    vis    = draw_boxes(img, boxes, cls_ids, confs)
    ax.imshow(vis); ax.axis('off')
    violations = [VIOLATION_CLASSES[c] for c in cls_ids if c in VIOLATION_CLASSES]
    ax.set_title('\n'.join(set(violations)) if violations else 'Compliant', fontsize=8)

plt.suptitle('Test-set Predictions', fontsize=14)
plt.tight_layout(); plt.show()

## 7 · Export Model (ONNX / TorchScript)
Export for deployment on edge devices or inference servers.

In [ ]:
export_model = YOLO(best_weights)

# ONNX (most compatible)
export_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)

# TorchScript (for PyTorch Mobile)
# export_model.export(format='torchscript', imgsz=IMG_SIZE)

print('Model exported successfully.')